# 01 — Exploração dos dados (EDA)

Competição Kaggle [Name That LLM](https://www.kaggle.com/competitions/name-that-llm).

Esse primeiro notebook aborda:
  1. Olhar geral sobre o problema. Estudar o objetivo definido.
  2. Obter e carregar os dados.
  3. Análise exploratória dos dados
     - Entender as características do dataset
     - Gerar Insights

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
KAGGLE_DIR = Path("/kaggle/input/name-that-llm")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path("../data")

## 1. Objetivo

- **Tarefa:** identificar qual de 18 LLMs escreveu cada texto (40–250 palavras).
- **Métrica:** balanced accuracy (média do recall por classe).
- **Referência:** chute aleatório ≈ 1/18 ≈ 5,6%.

## 2. Obter e carregar os dados

In [ ]:
name_llm = pd.read_csv(DATA_DIR / "train.csv")

## 3. Análise exploratória dos dados

### 3.1 Resumo dos dados

In [ ]:
info = pd.DataFrame({
    'Coluna': name_llm.columns,
    'Tipo': name_llm.dtypes.values,
    'Nao_Nulos': name_llm.notnull().sum().values,
    'Nulos': name_llm.isnull().sum().values,
    'Pct_Nulos (%)': (name_llm.isnull().mean() * 100).round(2).values,
    'Uso_Memoria (Bytes)': name_llm.memory_usage(deep=True, index=False).values
})
info

### 3.2 Forma dos dados

In [ ]:
name_llm.head(5)

### 3.3 Balanceamento dos dados

In [ ]:
model_prompt = pd.DataFrame({
    "Qtd_Prompts": name_llm["model"].value_counts()
})
model_prompt

In [ ]:
prompt_model = pd.DataFrame({
    "Qtd_Models": name_llm["prompt_id"].value_counts()
})
prompt_model.describe()

### 3.4 Insights

A partir dos resultados acima, pode-se considerar algumas afirmações:
- Os dados tem completude de 100%.
- Os dados são quase completamente balanceados.
  - Cada das categorias de classificação possui 150 prompts aplicados.
  - Cada um dos 150 prompts foram aplicados em cada um dos 18 modelos.
- A coluna "text" é o X.
  - É necessário fazer a devida conversão dos valores para ser possível alimentar o modelo adequadamente.
- A coluna "model" é o Y. 
- A coluna "prompt_id" deve ser um fator de agrupamento.
  - Se textos do mesmo prompt caírem no treino e na validação, o modelo é avaliado em prompts que já viu e a nota sai otimista (vazamento). O teste escondido deve usar prompts novos, então a validação precisa separar por prompt.
- A coluna "id" não tem valor como feature.